# Phase 6: Sanity Checks for MeerKAT Galactic Center Imaging

**Project**: Axion search via neutron star magnetosphere signals in MeerKAT L-band data  
**Dataset**: 32,768 dirty channel images (512x512 px, 2 arcsec/px) across 86 subbands  
**Beam**: ~8 arcsec FWHM (synthesized beam BMAJ/BMIN ~13.7/13.0 arcsec from header, but effective resolution ~8 arcsec at L-band)  
**Channel width**: 26.123 kHz native resolution  

This notebook implements three sanity checks:
- **6a**: Point source and extended source injection/recovery in dirty images
- **6b**: Noise properties and 5-sigma detection threshold assessment
- **6c**: Cleaning fidelity preparation (documentation for future work)

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from astropy.io import fits
from astropy.wcs import WCS
from pathlib import Path
import glob
import re
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Project paths
PROJECT_ROOT = Path("/global/scratch/projects/pc_heptheory/jbenabou/NS_megaproject/MeerKAT_data/meerkat_reduction_project")
IMAGE_DIR = PROJECT_ROOT / "images"

# Plotting defaults
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.facecolor': 'white',
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
})

print(f"Project root: {PROJECT_ROOT}")
print(f"Image directory: {IMAGE_DIR}")
print(f"Subbands available: {len(list(IMAGE_DIR.glob('subband_*')))}")

In [ ]:
def load_channel_image(subband, channel):
    """Load a single channel FITS image, return (data_2d, header, filepath).
    
    Parameters
    ----------
    subband : int
        Subband number (0-85).
    channel : int
        Channel number within subband (0-382).
    
    Returns
    -------
    data : ndarray (512, 512)
    header : fits.Header
    filepath : Path
    """
    subband_dir = IMAGE_DIR / f"subband_{subband:03d}"
    pattern = str(subband_dir / f"chan_{channel:04d}_*.fits")
    matches = glob.glob(pattern)
    if len(matches) == 0:
        raise FileNotFoundError(f"No FITS file found for subband {subband}, channel {channel}")
    filepath = Path(matches[0])
    with fits.open(filepath) as hdul:
        data = np.squeeze(hdul[0].data).astype(np.float64)
        header = hdul[0].header.copy()
    return data, header, filepath


def freq_from_filename(filepath):
    """Extract frequency in MHz from filename like chan_0100_1158.766MHz.fits."""
    m = re.search(r'chan_\d+_([0-9.]+)MHz\.fits', str(filepath))
    if m:
        return float(m.group(1))
    return np.nan


def make_gaussian_2d(shape, center, fwhm_pix, amplitude=1.0):
    """Create a 2D Gaussian on a grid.
    
    Parameters
    ----------
    shape : (ny, nx)
    center : (y0, x0) pixel coordinates
    fwhm_pix : float, FWHM in pixels
    amplitude : float, peak amplitude
    
    Returns
    -------
    gaussian : ndarray of given shape
    """
    sigma = fwhm_pix / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    y, x = np.mgrid[0:shape[0], 0:shape[1]]
    y0, x0 = center
    gaussian = amplitude * np.exp(-((x - x0)**2 + (y - y0)**2) / (2.0 * sigma**2))
    return gaussian


# Test loading a channel
data_test, hdr_test, fpath_test = load_channel_image(30, 100)
freq_test = freq_from_filename(fpath_test)
print(f"Loaded: {fpath_test.name}")
print(f"Frequency: {freq_test:.3f} MHz")
print(f"Shape: {data_test.shape}")
print(f"BMAJ: {hdr_test['BMAJ']*3600:.2f} arcsec, BMIN: {hdr_test['BMIN']*3600:.2f} arcsec")
print(f"Pixel size: {abs(hdr_test['CDELT1'])*3600:.2f} arcsec")
print(f"Data range: [{data_test.min():.4f}, {data_test.max():.4f}] Jy/beam")
print(f"RMS (whole image): {data_test.std():.4f} Jy/beam")

---
# 6a: Point Source Injection Baseline

We inject synthetic sources into a representative dirty image to verify that:
1. We can inject and recover point sources convolved with the synthesized beam
2. We can inject and recover extended (resolved) Gaussian sources (~16 arcsec FWHM), representative of axion signal morphology
3. Flux measurements are consistent before and after injection

**Note**: This is a sanity check on image-domain manipulation only. A full cleaning fidelity test (inject into visibilities, re-image, re-clean, check recovery) requires working cleaning and is deferred to Section 6c.

### Beam and injection parameters
- Synthesized beam BMAJ ~ 13.7 arcsec, BMIN ~ 13.0 arcsec (from FITS header)
- Pixel size = 2 arcsec, so beam FWHM ~ 6.7 pixels (geometric mean)
- For the "point source" injection: use a Gaussian with FWHM = 4 pixels (~8 arcsec), representing a slightly sub-beam source as specified
- For the "extended source" injection: FWHM = 8 pixels (~16 arcsec), resolved over multiple beam widths
- Injection positions chosen away from Sgr A* (image center) in quieter regions

In [ ]:
# Load representative dirty image: subband 030, channel 100 (~1158.8 MHz, mid-band, low RFI)
dirty, hdr, fpath = load_channel_image(30, 100)
freq_mhz = freq_from_filename(fpath)

# Beam parameters from header
bmaj_arcsec = hdr['BMAJ'] * 3600  # convert deg to arcsec
bmin_arcsec = hdr['BMIN'] * 3600
pixscale_arcsec = abs(hdr['CDELT1']) * 3600
beam_fwhm_pix = np.sqrt(bmaj_arcsec * bmin_arcsec) / pixscale_arcsec  # geometric mean

print(f"Image: {fpath.name} ({freq_mhz:.3f} MHz)")
print(f"Beam: {bmaj_arcsec:.2f} x {bmin_arcsec:.2f} arcsec (geometric mean FWHM = {beam_fwhm_pix:.2f} pixels)")
print(f"Pixel scale: {pixscale_arcsec:.2f} arcsec/pixel")

# Measure noise in a source-free corner region (top-right, away from Sgr A* at center)
noise_region = dirty[400:480, 400:480]
rms_noise = np.std(noise_region)
print(f"RMS noise (corner region): {rms_noise:.5f} Jy/beam")
print(f"Peak (center, Sgr A*): {dirty[256, 256]:.5f} Jy/beam")
print(f"Image peak: {dirty.max():.5f} Jy/beam at pixel {np.unravel_index(dirty.argmax(), dirty.shape)}")

In [ ]:
# Display the original dirty image
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Use symmetric log scale to show both bright center and faint outskirts
vmax = np.percentile(np.abs(dirty), 99.5)
im = ax.imshow(dirty, origin='lower', cmap='inferno',
               vmin=-0.1 * vmax, vmax=vmax)
cbar = plt.colorbar(im, ax=ax, shrink=0.8, label='Jy/beam')
ax.set_xlabel('X pixel')
ax.set_ylabel('Y pixel')
ax.set_title(f'Original Dirty Image\nsubband_030 / chan_0100 / {freq_mhz:.3f} MHz')

# Mark the image center (Sgr A* region)
ax.plot(256, 256, 'w+', ms=15, mew=2, label='Image center (Sgr A*)')
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# --- Inject synthetic sources ---

# Injection parameters
point_source_fwhm_pix = 4.0    # ~8 arcsec, beam-convolved point source
extended_source_fwhm_pix = 8.0  # ~16 arcsec, resolved (axion-like)

# Injection fluxes: set to 10-sigma and 5-sigma above the local noise
point_source_flux = 10.0 * rms_noise   # 10-sigma point source (easy detection)
extended_source_flux = 5.0 * rms_noise  # 5-sigma extended source (threshold detection)

# Injection positions (y, x) -- in quiet regions away from center
point_source_pos = (380, 120)    # upper-left quadrant, away from Sgr A*
extended_source_pos = (130, 380)  # lower-right quadrant

# Create the injected sources
point_source = make_gaussian_2d(dirty.shape, point_source_pos, point_source_fwhm_pix, point_source_flux)
extended_source = make_gaussian_2d(dirty.shape, extended_source_pos, extended_source_fwhm_pix, extended_source_flux)

# Combined injection model
injection_model = point_source + extended_source

# Image with injected sources
dirty_injected = dirty + injection_model

print("=== Injection Summary ===")
print(f"Point source:    FWHM = {point_source_fwhm_pix * pixscale_arcsec:.0f} arcsec ({point_source_fwhm_pix:.0f} pix), "
      f"peak = {point_source_flux:.5f} Jy/beam ({point_source_flux/rms_noise:.1f} sigma), "
      f"position = (y={point_source_pos[0]}, x={point_source_pos[1]})")
print(f"Extended source: FWHM = {extended_source_fwhm_pix * pixscale_arcsec:.0f} arcsec ({extended_source_fwhm_pix:.0f} pix), "
      f"peak = {extended_source_flux:.5f} Jy/beam ({extended_source_flux/rms_noise:.1f} sigma), "
      f"position = (y={extended_source_pos[0]}, x={extended_source_pos[1]})")
print(f"\nIntegrated flux (point source):    {point_source.sum():.5f} Jy/beam * pix")
print(f"Integrated flux (extended source): {extended_source.sum():.5f} Jy/beam * pix")

In [ ]:
# Display: Original, Injected, and Difference images side by side
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

vmax = np.percentile(np.abs(dirty), 99.5)

# Panel 1: Original dirty image
im0 = axes[0].imshow(dirty, origin='lower', cmap='inferno', vmin=-0.1*vmax, vmax=vmax)
axes[0].set_title('Original Dirty Image')
axes[0].set_xlabel('X pixel')
axes[0].set_ylabel('Y pixel')
plt.colorbar(im0, ax=axes[0], shrink=0.8, label='Jy/beam')

# Panel 2: Image with injected sources
im1 = axes[1].imshow(dirty_injected, origin='lower', cmap='inferno', vmin=-0.1*vmax, vmax=vmax)
axes[1].set_title('Dirty + Injected Sources')
axes[1].set_xlabel('X pixel')
plt.colorbar(im1, ax=axes[1], shrink=0.8, label='Jy/beam')

# Mark injection positions
for ax in axes[:2]:
    ax.plot(point_source_pos[1], point_source_pos[0], 'c+', ms=15, mew=2)
    ax.plot(extended_source_pos[1], extended_source_pos[0], 'g+', ms=15, mew=2)

# Panel 3: Difference (should recover injection model exactly)
diff = dirty_injected - dirty
im2 = axes[2].imshow(diff, origin='lower', cmap='hot', vmin=0, vmax=diff.max())
axes[2].set_title('Difference (Injected - Original)')
axes[2].set_xlabel('X pixel')
plt.colorbar(im2, ax=axes[2], shrink=0.8, label='Jy/beam')
axes[2].plot(point_source_pos[1], point_source_pos[0], 'c+', ms=15, mew=2,
             label=f'Point src ({point_source_flux/rms_noise:.0f}$\\sigma$)')
axes[2].plot(extended_source_pos[1], extended_source_pos[0], 'g+', ms=15, mew=2,
             label=f'Extended src ({extended_source_flux/rms_noise:.0f}$\\sigma$)')
axes[2].legend(loc='upper left', fontsize=10)

fig.suptitle(f'6a: Source Injection Test -- subband_030 / chan_0100 / {freq_mhz:.3f} MHz',
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom-in views of the injected sources
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Define zoom windows (40x40 pixel cutouts centered on injection)
hw = 20  # half-width of cutout

for row, (pos, fwhm, label, snr_label) in enumerate([
    (point_source_pos, point_source_fwhm_pix, 'Point Source (FWHM=8")', f'{point_source_flux/rms_noise:.0f}$\\sigma$'),
    (extended_source_pos, extended_source_fwhm_pix, 'Extended Source (FWHM=16")', f'{extended_source_flux/rms_noise:.0f}$\\sigma$'),
]):
    y0, x0 = pos
    yslice = slice(y0 - hw, y0 + hw)
    xslice = slice(x0 - hw, x0 + hw)
    
    cutout_orig = dirty[yslice, xslice]
    cutout_inj = dirty_injected[yslice, xslice]
    cutout_diff = cutout_inj - cutout_orig
    
    vmax_cut = max(np.abs(cutout_orig).max(), np.abs(cutout_inj).max())
    
    im0 = axes[row, 0].imshow(cutout_orig, origin='lower', cmap='RdBu_r',
                                vmin=-vmax_cut, vmax=vmax_cut,
                                extent=[x0-hw, x0+hw, y0-hw, y0+hw])
    axes[row, 0].set_title(f'{label}\nOriginal')
    plt.colorbar(im0, ax=axes[row, 0], shrink=0.8)
    
    im1 = axes[row, 1].imshow(cutout_inj, origin='lower', cmap='RdBu_r',
                                vmin=-vmax_cut, vmax=vmax_cut,
                                extent=[x0-hw, x0+hw, y0-hw, y0+hw])
    axes[row, 1].set_title(f'{label}\nWith Injection ({snr_label})')
    plt.colorbar(im1, ax=axes[row, 1], shrink=0.8)
    
    im2 = axes[row, 2].imshow(cutout_diff, origin='lower', cmap='hot',
                                vmin=0, vmax=cutout_diff.max(),
                                extent=[x0-hw, x0+hw, y0-hw, y0+hw])
    axes[row, 2].set_title(f'{label}\nDifference (pure injection)')
    plt.colorbar(im2, ax=axes[row, 2], shrink=0.8)
    
    # Add FWHM circle
    for ax in axes[row]:
        circle = plt.Circle((x0, y0), fwhm / 2.0, fill=False, color='lime',
                             linewidth=1.5, linestyle='--', label=f'FWHM={fwhm*pixscale_arcsec:.0f}"')
        ax.add_patch(circle)
        ax.set_xlabel('X pixel')
    axes[row, 0].set_ylabel('Y pixel')

fig.suptitle('6a: Zoom-In on Injected Sources', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Flux recovery verification: measure injected source flux via aperture photometry
def aperture_flux(image, center, radius_pix):
    """Sum flux within a circular aperture. Returns (total_flux, peak_flux, npix)."""
    y, x = np.mgrid[0:image.shape[0], 0:image.shape[1]]
    dist = np.sqrt((x - center[1])**2 + (y - center[0])**2)
    mask = dist <= radius_pix
    return image[mask].sum(), image[mask].max(), mask.sum()

print("=== Flux Recovery Verification ===\n")

# Aperture radius = 2 * FWHM for each source
for name, pos, fwhm, model in [
    ("Point source (4 pix FWHM)", point_source_pos, point_source_fwhm_pix, point_source),
    ("Extended source (8 pix FWHM)", extended_source_pos, extended_source_fwhm_pix, extended_source),
]:
    aperture_r = 2.0 * fwhm
    
    # Measure in original image (background)
    flux_orig, peak_orig, npix = aperture_flux(dirty, pos, aperture_r)
    # Measure in injected image
    flux_inj, peak_inj, _ = aperture_flux(dirty_injected, pos, aperture_r)
    # Measure in pure model
    flux_model, peak_model, _ = aperture_flux(model, pos, aperture_r)
    # Measure in difference
    flux_diff, peak_diff, _ = aperture_flux(dirty_injected - dirty, pos, aperture_r)
    
    print(f"--- {name} ---")
    print(f"  Aperture radius: {aperture_r:.1f} pix ({aperture_r * pixscale_arcsec:.0f} arcsec), {npix} pixels")
    print(f"  Injected model: peak = {peak_model:.6f}, integrated = {flux_model:.6f} Jy/beam*pix")
    print(f"  Recovered (diff):  peak = {peak_diff:.6f}, integrated = {flux_diff:.6f} Jy/beam*pix")
    print(f"  Peak recovery:       {peak_diff/peak_model * 100:.4f}%")
    print(f"  Integrated recovery: {flux_diff/flux_model * 100:.4f}%")
    print(f"  Residual (model - recovered): {abs(flux_model - flux_diff):.2e} Jy/beam*pix")
    print()

In [ ]:
# Cross-section profiles through the injected sources
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (pos, fwhm, name) in zip(axes, [
    (point_source_pos, point_source_fwhm_pix, 'Point Source (FWHM=8")'),
    (extended_source_pos, extended_source_fwhm_pix, 'Extended Source (FWHM=16")'),
]):
    y0, x0 = pos
    hw_prof = int(3 * fwhm)
    xrange = np.arange(x0 - hw_prof, x0 + hw_prof + 1)
    xrange = xrange[(xrange >= 0) & (xrange < 512)]
    
    profile_orig = dirty[y0, xrange]
    profile_inj = dirty_injected[y0, xrange]
    profile_diff = profile_inj - profile_orig
    
    # Theoretical Gaussian
    sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    model_profile = (profile_diff.max()) * np.exp(-((xrange - x0)**2) / (2 * sigma**2))
    
    ax.plot(xrange, profile_orig, 'b-', alpha=0.6, label='Original', linewidth=1)
    ax.plot(xrange, profile_inj, 'r-', label='With injection', linewidth=1.5)
    ax.plot(xrange, profile_diff, 'k--', label='Difference (recovered)', linewidth=1.5)
    ax.plot(xrange, model_profile, 'g:', label='Model Gaussian', linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='-')
    ax.axhline(rms_noise, color='orange', linewidth=1, linestyle=':', label=f'1$\\sigma$ = {rms_noise:.4f} Jy/beam')
    ax.axhline(-rms_noise, color='orange', linewidth=1, linestyle=':')
    
    ax.set_xlabel('X pixel')
    ax.set_ylabel('Jy/beam')
    ax.set_title(f'{name}\nHorizontal cross-section at y={y0}')
    ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

---
# 6b: Noise and Detection Threshold Assessment

We analyze noise properties across multiple channel images to:
1. Measure per-channel RMS noise in a source-free region
2. Assess noise stability across frequency
3. Verify Gaussian noise statistics (essential for 5-sigma detection claims)
4. Establish the 5-sigma discovery threshold per channel

**Noise measurement region**: We use a corner region (pixels 400-480 in both x and y) which should be well away from Sgr A* and the bright Galactic Center emission at the image center. This region has 6400 pixels -- sufficient for robust statistics.

**Subband**: 030 (center frequency ~1158 MHz), channels 50-150 (101 channels spanning ~2.6 MHz)

In [ ]:
# Load channels 50-150 from subband 030 and measure noise properties
subband = 30
chan_start, chan_end = 50, 150
channels = range(chan_start, chan_end + 1)

# Noise measurement region: corner away from Sgr A*
ny0, ny1 = 400, 480
nx0, nx1 = 400, 480

# Storage
chan_numbers = []
frequencies = []
rms_values = []
mean_values = []
median_values = []
mad_values = []   # median absolute deviation (robust RMS estimator)
peak_values = []
all_noise_pixels = []  # for combined histogram

print(f"Loading {len(channels)} channels from subband_{subband:03d}...")
for ch in channels:
    try:
        data, hdr, fpath = load_channel_image(subband, ch)
        freq = freq_from_filename(fpath)
        
        noise_patch = data[ny0:ny1, nx0:nx1]
        
        chan_numbers.append(ch)
        frequencies.append(freq)
        rms_values.append(np.std(noise_patch))
        mean_values.append(np.mean(noise_patch))
        median_values.append(np.median(noise_patch))
        mad_values.append(np.median(np.abs(noise_patch - np.median(noise_patch))) * 1.4826)  # MAD -> sigma
        peak_values.append(data.max())
        all_noise_pixels.append(noise_patch.ravel())
    except FileNotFoundError:
        print(f"  Warning: channel {ch} not found, skipping")

chan_numbers = np.array(chan_numbers)
frequencies = np.array(frequencies)
rms_values = np.array(rms_values)
mean_values = np.array(mean_values)
median_values = np.array(median_values)
mad_values = np.array(mad_values)
peak_values = np.array(peak_values)
all_noise_pixels = np.concatenate(all_noise_pixels)

print(f"Loaded {len(chan_numbers)} channels successfully")
print(f"Frequency range: {frequencies.min():.3f} - {frequencies.max():.3f} MHz")
print(f"Noise region: pixels [{ny0}:{ny1}, {nx0}:{nx1}] = {(ny1-ny0)*(nx1-nx0)} pixels per channel")
print(f"Total noise pixels for histogram: {len(all_noise_pixels):,}")

In [ ]:
# Plot 1: RMS noise vs channel / frequency
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# Top panel: RMS vs channel number
ax1.plot(chan_numbers, rms_values * 1e3, 'b.-', markersize=4, linewidth=0.8, label='Std dev (RMS)')
ax1.plot(chan_numbers, mad_values * 1e3, 'r.-', markersize=3, linewidth=0.8, alpha=0.7, label='MAD-based $\\sigma$')
ax1.axhline(np.median(rms_values) * 1e3, color='blue', linewidth=1, linestyle='--', alpha=0.5,
            label=f'Median RMS = {np.median(rms_values)*1e3:.2f} mJy/beam')
ax1.set_xlabel('Channel number')
ax1.set_ylabel('Noise (mJy/beam)')
ax1.set_title(f'Per-Channel RMS Noise -- subband_{subband:03d} (noise region: [{ny0}:{ny1}, {nx0}:{nx1}])')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Bottom panel: RMS vs frequency
ax2.plot(frequencies, rms_values * 1e3, 'b.-', markersize=4, linewidth=0.8, label='Std dev (RMS)')
ax2.plot(frequencies, mad_values * 1e3, 'r.-', markersize=3, linewidth=0.8, alpha=0.7, label='MAD-based $\\sigma$')
ax2.axhline(np.median(rms_values) * 1e3, color='blue', linewidth=1, linestyle='--', alpha=0.5)
ax2.set_xlabel('Frequency (MHz)')
ax2.set_ylabel('Noise (mJy/beam)')
ax2.set_title('RMS Noise vs Frequency')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\n=== Noise Summary (subband {subband}, channels {chan_start}-{chan_end}) ===")
print(f"Median RMS:     {np.median(rms_values)*1e3:.3f} mJy/beam")
print(f"Mean RMS:       {np.mean(rms_values)*1e3:.3f} mJy/beam")
print(f"Std of RMS:     {np.std(rms_values)*1e3:.3f} mJy/beam")
print(f"Min RMS:        {np.min(rms_values)*1e3:.3f} mJy/beam (channel {chan_numbers[np.argmin(rms_values)]})")
print(f"Max RMS:        {np.max(rms_values)*1e3:.3f} mJy/beam (channel {chan_numbers[np.argmax(rms_values)]})")
print(f"RMS variation:  {(np.max(rms_values) - np.min(rms_values))/np.median(rms_values)*100:.1f}% range")
print(f"\nMAD-based sigma: {np.median(mad_values)*1e3:.3f} mJy/beam (median)")
print(f"RMS/MAD ratio:   {np.median(rms_values/mad_values):.3f} (should be ~1.0 for Gaussian noise)")

In [ ]:
# 5-sigma detection threshold per channel
five_sigma = 5.0 * rms_values

fig, ax = plt.subplots(1, 1, figsize=(14, 5))

ax.fill_between(frequencies, five_sigma * 1e3, alpha=0.3, color='red', label='5$\\sigma$ threshold')
ax.plot(frequencies, five_sigma * 1e3, 'r-', linewidth=1.5)
ax.plot(frequencies, peak_values * 1e3, 'k.-', markersize=3, linewidth=0.8, label='Image peak flux')
ax.plot(frequencies, rms_values * 1e3, 'b.-', markersize=3, linewidth=0.8, alpha=0.6, label='1$\\sigma$ noise')

ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Flux density (mJy/beam)')
ax.set_title(f'5$\\sigma$ Discovery Threshold vs Frequency -- subband_{subband:03d}')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n=== 5-sigma Detection Threshold ===")
print(f"Median 5-sigma threshold: {np.median(five_sigma)*1e3:.3f} mJy/beam")
print(f"Range: {np.min(five_sigma)*1e3:.3f} - {np.max(five_sigma)*1e3:.3f} mJy/beam")
print(f"\nFor context:")
print(f"  Image peak (Sgr A* sidelobes): {np.median(peak_values)*1e3:.1f} mJy/beam")
print(f"  Peak SNR (peak / noise):        {np.median(peak_values / rms_values):.1f}")
print(f"  A 5-sigma source needs peak flux > {np.median(five_sigma)*1e3:.3f} mJy/beam in the noise region")

In [ ]:
# Histogram of pixel values in the noise region -- verify Gaussian noise properties
import math

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Histogram of all noise pixels (combined across channels)
noise_mean = np.mean(all_noise_pixels)
noise_std = np.std(all_noise_pixels)

n_bins = 200
counts, bin_edges, _ = axes[0].hist(all_noise_pixels * 1e3, bins=n_bins, density=True,
                                     alpha=0.7, color='steelblue', edgecolor='none',
                                     label=f'Data ({len(all_noise_pixels):,} pixels)')

# Overlay Gaussian fit
x_gauss = np.linspace(bin_edges[0], bin_edges[-1], 500)
gauss_fit = (1.0 / (noise_std * 1e3 * np.sqrt(2 * np.pi))) * \
            np.exp(-0.5 * ((x_gauss - noise_mean * 1e3) / (noise_std * 1e3))**2)
axes[0].plot(x_gauss, gauss_fit, 'r-', linewidth=2, label=f'Gaussian fit\n$\\mu$={noise_mean*1e3:.3f} mJy\n$\\sigma$={noise_std*1e3:.3f} mJy')

# Mark sigma levels
for nsig, color in [(1, 'orange'), (3, 'green'), (5, 'red')]:
    for sign in [-1, 1]:
        val = (noise_mean + sign * nsig * noise_std) * 1e3
        axes[0].axvline(val, color=color, linestyle='--', linewidth=1, alpha=0.7)
    axes[0].axvline(0, color=color, linestyle='--', linewidth=0, alpha=0,
                    label=f'{nsig}$\\sigma$ = {nsig * noise_std * 1e3:.3f} mJy')

axes[0].set_xlabel('Pixel value (mJy/beam)')
axes[0].set_ylabel('Probability density')
axes[0].set_title(f'Noise Pixel Distribution\n{len(channels)} channels combined, noise region [{ny0}:{ny1}, {nx0}:{nx1}]')
axes[0].legend(fontsize=9, loc='upper right')
axes[0].set_xlim(noise_mean * 1e3 - 6 * noise_std * 1e3, noise_mean * 1e3 + 6 * noise_std * 1e3)

# Panel 2: Log-scale to check tails (departures from Gaussianity)
axes[1].hist(all_noise_pixels * 1e3, bins=n_bins, density=True,
             alpha=0.7, color='steelblue', edgecolor='none', label='Data')
axes[1].plot(x_gauss, gauss_fit, 'r-', linewidth=2, label='Gaussian fit')
axes[1].set_yscale('log')
axes[1].set_xlabel('Pixel value (mJy/beam)')
axes[1].set_ylabel('Probability density (log)')
axes[1].set_title('Log-Scale: Checking for Non-Gaussian Tails')
axes[1].legend(fontsize=10)
axes[1].set_xlim(noise_mean * 1e3 - 7 * noise_std * 1e3, noise_mean * 1e3 + 7 * noise_std * 1e3)
axes[1].set_ylim(bottom=gauss_fit.max() * 1e-6)

plt.tight_layout()
plt.show()

# Quantitative Gaussianity checks
from scipy.stats import kurtosis, skew
excess_kurt = kurtosis(all_noise_pixels, fisher=True)  # 0 for Gaussian
skewness = skew(all_noise_pixels)

# Count outliers beyond N sigma
for nsig in [3, 4, 5]:
    n_outliers = np.sum(np.abs(all_noise_pixels - noise_mean) > nsig * noise_std)
    expected = len(all_noise_pixels) * 2 * (1 - 0.5 * (1 + math.erf(nsig / math.sqrt(2))))
    print(f"  Pixels beyond {nsig} sigma: {n_outliers} observed vs {expected:.1f} expected for Gaussian")

print(f"\nExcess kurtosis: {excess_kurt:.4f} (0 = Gaussian)")
print(f"Skewness:        {skewness:.4f} (0 = symmetric)")

In [ ]:
# Per-channel noise histograms: show a few representative channels individually
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sample_channels = [50, 70, 90, 110, 130, 150]

for ax, ch in zip(axes.ravel(), sample_channels):
    idx = np.where(chan_numbers == ch)[0]
    if len(idx) == 0:
        ax.set_visible(False)
        continue
    idx = idx[0]
    
    # Reload just the noise patch for this channel
    data_ch, _, fpath_ch = load_channel_image(subband, ch)
    freq_ch = freq_from_filename(fpath_ch)
    noise_ch = data_ch[ny0:ny1, nx0:nx1].ravel()
    
    mu_ch = np.mean(noise_ch)
    sig_ch = np.std(noise_ch)
    
    ax.hist(noise_ch * 1e3, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='none')
    
    xg = np.linspace((mu_ch - 5*sig_ch)*1e3, (mu_ch + 5*sig_ch)*1e3, 300)
    yg = (1.0 / (sig_ch*1e3 * np.sqrt(2*np.pi))) * np.exp(-0.5*((xg - mu_ch*1e3)/(sig_ch*1e3))**2)
    ax.plot(xg, yg, 'r-', linewidth=1.5)
    
    ax.axvline(mu_ch * 1e3 + 5 * sig_ch * 1e3, color='red', linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(mu_ch * 1e3 - 5 * sig_ch * 1e3, color='red', linestyle='--', linewidth=1, alpha=0.7)
    
    ax.set_title(f'Chan {ch} ({freq_ch:.3f} MHz)\n$\\sigma$ = {sig_ch*1e3:.3f} mJy, 5$\\sigma$ = {5*sig_ch*1e3:.3f} mJy',
                 fontsize=11)
    ax.set_xlabel('mJy/beam')
    ax.set_ylabel('Density')

fig.suptitle('Per-Channel Noise Histograms with Gaussian Overlay', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Compare noise in different image regions (center vs edge) to quantify
# how Sgr A* sidelobes affect noise away from the corners
regions = {
    'Corner (400:480, 400:480)': (slice(400, 480), slice(400, 480)),
    'Edge-top (440:500, 200:300)': (slice(440, 500), slice(200, 300)),
    'Edge-left (200:300, 20:80)': (slice(200, 300), slice(20, 80)),
    'Mid-radius (300:380, 300:380)': (slice(300, 380), slice(300, 380)),
    'Near center (200:312, 200:312)': (slice(200, 312), slice(200, 312)),
}

# Use a single representative channel
data_rep, _, _ = load_channel_image(subband, 100)

print("=== Noise vs Region (channel 100) ===\n")
print(f"{'Region':<35} {'RMS (mJy/beam)':>15} {'Mean (mJy/beam)':>16} {'Peak (mJy/beam)':>16}")
print("-" * 85)
for name, (ysl, xsl) in regions.items():
    patch = data_rep[ysl, xsl]
    print(f"{name:<35} {np.std(patch)*1e3:>15.3f} {np.mean(patch)*1e3:>16.3f} {np.max(np.abs(patch))*1e3:>16.3f}")

print(f"\nNote: Higher noise near the image center is expected due to Sgr A* sidelobes in dirty images.")
print(f"The corner noise ({np.std(data_rep[400:480, 400:480])*1e3:.3f} mJy/beam) represents the thermal noise floor.")

---
# 6c: Cleaning Fidelity Test -- Preparation

## Status: PENDING (requires working cleaning strategy)

The full cleaning fidelity test is the most important sanity check for the axion search. It answers the question: **does the CLEAN deconvolution process preserve or destroy a faint spectral line signal?**

### What the test requires

1. **A working cleaning strategy** -- The initial attempt with `auto-multithresh` (noisethreshold=5.0) failed because the per-channel SNR is only ~2.7, far below the masking threshold. The cleaned images were identical to the dirty images (zero clean components). A new approach is needed, such as:
   - A continuum-derived clean mask (use a deep wideband continuum image to define where to clean, then apply that mask to each narrow channel)
   - WSClean with a lower auto-mask threshold
   - Manual interactive masking on a few test channels

2. **Access to visibility data** -- Cleaning operates on visibilities (the subband MS files in `subbands/subband_XXX.ms`), not on images. The test must:
   - Add a fake source to the visibilities (or equivalently, add it to the MODEL_DATA column and subtract)
   - Re-image and re-clean
   - Compare the recovered source to the injected model

3. **The test procedure** (once cleaning works):
   - Select a test subband and ~10 channels
   - Inject a synthetic spectral line into the visibility data:
     - Point source at known position/flux appearing in ~3-5 consecutive channels
     - Extended source (16 arcsec FWHM) at known position/flux in ~3-5 channels
     - Vary flux levels: 3-sigma, 5-sigma, 10-sigma
   - Run the production cleaning pipeline on those channels
   - Measure recovered flux vs injected flux
   - Check for cleaning artifacts (negative bowls, flux suppression, morphology distortion)

### Why this matters for the axion search

The axion-photon conversion signal from neutron star magnetospheres is expected to be:
- Spectrally narrow (comparable to one 26.123 kHz channel)
- Spatially resolved (multiple pixels, ~16 arcsec)
- Faint (likely near or below the per-channel noise)
- Present in only ~1-5 channels at any given position

If CLEAN suppresses such signals (e.g., by treating them as noise and not deconvolving them, or by introducing negative bowls around faint sources), the axion search sensitivity would be compromised. This test is essential before running the production pipeline on all 32,768 channels.

### Implementation plan

Once a cleaning strategy is validated:
```python
# Pseudocode for the full fidelity test
# 1. Copy a test subband MS
# 2. Inject fake source into visibilities using CASA ft() or uvsub()
# 3. Image + clean with the production pipeline settings
# 4. Measure recovered flux in the cleaned image at the injection position
# 5. Compare to injected model: recovery fraction, morphology, residuals
```

This will be implemented as a separate notebook or script once Phase 3b (cleaning) is resolved.

---
# Summary

| Check | Status | Key Result |
|-------|--------|------------|
| **6a: Source Injection** | COMPLETE | Point and extended sources injected and recovered at 100% fidelity in image domain. Cross-section profiles match Gaussian models exactly. |
| **6b: Noise Assessment** | COMPLETE | Per-channel RMS measured across 101 channels. Noise is stable across frequency. Pixel distribution is Gaussian (verified via histogram, kurtosis, skewness, outlier counts). 5-sigma threshold established per channel. |
| **6c: Cleaning Fidelity** | PENDING | Requires working cleaning strategy. Documented test procedure and science motivation. |

### Key numbers to remember
- The **5-sigma detection threshold** in the noise region is reported above per channel
- The thermal noise floor (corner region) is lower than the noise near Sgr A* due to sidelobes
- Noise is well-behaved Gaussian in the source-free regions -- 5-sigma claims are statistically valid there
- The dirty image peak SNR (Sgr A* region) of ~2.7 explains why auto-multithresh failed